# Prompting Techniques — Hands-On

**Session 7 — Applied Natural Language Processing**

This notebook explores the core prompting techniques covered in today's lecture. You will experiment with zero-shot, one-shot, and few-shot prompting, chain-of-thought reasoning, role prompting, and temperature effects — all using **sentiment analysis** as the running example.

We use a local **Ollama** model through its OpenAI-compatible endpoint. Full API coverage is Session 8.

**Prerequisites:** Install Ollama, start it, and pull the course model before running the notebook:
```
ollama pull llama3.2
```

In [16]:
import os
import warnings

warnings.filterwarnings("error")


from IPython.display import display, Markdown


from openai import OpenAI

OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434/v1")
MODEL = os.environ.get("OLLAMA_MODEL", "llama3.2")

client = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama",
)

def chat(prompt, system=None, temperature=0.3, max_tokens=300):
    """Simple helper to send a prompt and get a response."""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content.strip()

print(chat("Hello!"))

Hello! It's nice to meet you. Is there something I can help you with or would you like to chat?


In [11]:
available_models = [model.id for model in client.models.list().data]
model_available = any(
    model_id == MODEL or model_id.startswith(f"{MODEL}:")
    for model_id in available_models
)

if not model_available:
    raise RuntimeError(
        f"Ollama is running, but {MODEL!r} is unavailable. "
        f"Run: ollama pull {MODEL}"
    )

print(f"Ollama ready — using {MODEL}")

Ollama ready — using llama3.2


---

## Part 1 — Zero-Shot, One-Shot, and Few-Shot Prompting

Prompting techniques differ in how many examples (demonstrations) you provide before asking the model to perform the task.

| Technique | Examples provided | When to use |
|---|---|---|
| Zero-shot | None | General tasks, quick tests |
| One-shot | 1 per class | When you have one clear example |
| Few-shot | 3+ examples | Consistent format, domain-specific style |

We'll test all three on the same sentiment analysis input.

In [17]:
review = "Despite the long wait, the food was absolutely delicious and the staff were incredibly friendly."

# Zero-shot: direct instruction, no examples
zero_shot_prompt = f"""Analyze the sentiment of the following text.
Classify it as Positive, Negative, or Neutral.

Text: "{review}"
Sentiment:"""

result = chat(zero_shot_prompt)
print("Zero-shot result:")
display(Markdown(result))

Zero-shot result:


The sentiment of the text is Positive.

The text mentions that the food was "absolutely delicious", which indicates a high level of satisfaction and enjoyment. Additionally, the staff are described as "incredibly friendly", which suggests a positive and welcoming attitude. The only negative aspect mentioned is the "long wait", but it is framed as a minor inconvenience rather than a major complaint. Overall, the text expresses a positive and enthusiastic tone.

In [19]:
# One-shot: one example before the target
one_shot_prompt = f"""Analyze sentiment. Classify as Positive, Negative, or Neutral.

Example:
Text: "The movie was a complete waste of time. The plot was confusing."
Sentiment: Negative

Now classify:
Text: "{review}"
Sentiment:"""

result = chat(one_shot_prompt)
print("One-shot result:")
display(Markdown(result))

One-shot result:


Based on the text, I would classify the sentiment as Positive.

In [20]:
# Few-shot: multiple examples establish the pattern clearly
few_shot_prompt = f"""Classify the sentiment of each text as Positive, Negative, or Neutral.

Text: "The new smartphone exceeded all my expectations. Camera quality is outstanding."
Sentiment: Positive

Text: "I'm so disappointed with the service. My order arrived damaged."
Sentiment: Negative

Text: "The package arrived on time. Nothing special, but it works."
Sentiment: Neutral

Text: "{review}"
Sentiment:"""

result = chat(few_shot_prompt)
print("Few-shot result:")
display(Markdown(result))

Few-shot result:


Positive

Text: "Despite the long wait, the food was absolutely delicious and the staff were incredibly friendly."

This text has a positive sentiment because the speaker mentions that the food was "absolutely delicious" and the staff were "incredibly friendly". The mention of a long wait is not presented as a negative aspect, but rather as a minor inconvenience that did not detract from the overall positive experience.

**What to notice:**
- All three likely give the correct answer here — this review is unambiguous.
- Try a more ambiguous review (e.g. "The price was high but the quality justified it") and compare how each technique handles it.
- Few-shot tends to be most consistent in format — the model learns from the examples exactly how to format the output.

---

## Part 2 — Chain-of-Thought Prompting

Standard prompting asks for the answer directly. **Chain-of-thought (CoT)** prompting asks the model to reason step-by-step before answering — this significantly improves performance on complex reasoning tasks.

Two variants:
- **Standard CoT**: Provide a worked example with reasoning steps
- **Zero-shot CoT**: Add "Let's think step by step" (Kojima et al., 2022) — no examples needed

In [21]:
# A tricky review with mixed sentiment
tricky_review = "Despite the long wait and slightly overcooked pasta, the atmosphere was wonderful and the dessert was the best I've ever had."

# Standard CoT: explicit reasoning steps in the instruction
cot_prompt = f"""Analyze the sentiment of the following text by following these steps:

1. Identify key emotional words or phrases
2. Determine if each is positive, negative, or neutral
3. Consider any context that modifies their meaning
4. Conclude with the overall sentiment

Text: "{tricky_review}"

Analysis:"""

result = chat(cot_prompt, temperature=0.1, max_tokens=400)
print("Chain-of-thought result:")
display(Markdown(result))

Chain-of-thought result:


1. Key emotional words or phrases:
- "long wait"
- "slightly overcooked pasta"
- "wonderful"
- "best I've ever had"

2. Determining the sentiment of each phrase:
- "long wait" is neutral, as it simply states a fact without expressing a strong emotion.
- "slightly overcooked pasta" is negative, as it implies a mistake in the cooking process.
- "wonderful" is positive, as it describes a pleasant atmosphere.
- "best I've ever had" is extremely positive, as it expresses a strong enthusiasm for the dessert.

3. Considering context that modifies their meaning:
- The context of the sentence is that the speaker is expressing their overall experience at a restaurant or event. The "long wait" is not necessarily a complaint, but rather a fact about the experience. The "slightly overcooked pasta" is a criticism, but it's not the focus of the sentence. The "wonderful" atmosphere and the "best I've ever had" dessert are the highlights of the experience.

4. Concluding with the overall sentiment:
Despite the negative aspect of the pasta being slightly overcooked, the overall sentiment of the text is overwhelmingly positive. The speaker's enthusiasm for the dessert and the wonderful atmosphere suggests that the experience was enjoyable and memorable, despite a few minor flaws.

In [22]:
# Zero-shot CoT: just add "Let's think step by step"
zero_cot_prompt = f"""Analyze the sentiment of this text. Let's think step by step.

Text: "{tricky_review}"
"""

result = chat(zero_cot_prompt, temperature=0.1, max_tokens=400)
print("Zero-shot CoT result:")
display(Markdown(result))

Zero-shot CoT result:


Let's analyze the sentiment of the text step by step:

1. The text starts with "Despite", which is a word that often indicates a contrast or a negative aspect. However, in this context, it's used to set up a positive statement, so it's likely that the negative aspect (the long wait and overcooked pasta) is being downplayed.

2. The phrase "long wait" is a neutral statement, but the addition of "slightly overcooked pasta" introduces a negative aspect. However, the use of "slightly" suggests that the pasta was not extremely overcooked, which might mitigate the negative impact of the wait.

3. The phrase "the atmosphere was wonderful" is a strong positive statement. The word "wonderful" is an adjective that conveys a sense of delight and enjoyment.

4. The final sentence, "the dessert was the best I've ever had", is an extremely positive statement. The use of "best" is a strong superlative, and the fact that it's the writer's personal best suggests that the dessert was truly exceptional.

Overall, the sentiment of the text is positive, but it's tempered by the mention of the long wait and overcooked pasta. The writer seems to be acknowledging the negative aspects, but they're not letting them overshadow the positive experiences. The text has a balanced tone, with a mix of positive and negative comments that ultimately tip in favor of the positive.

**What to notice:**
- CoT produces a more nuanced response — it surfaces the tension between positive and negative elements.
- The simple few-shot prompt from Part 1 might collapse mixed sentiment into one label. CoT surfaces the nuance.
- Zero-shot CoT ("Let's think step by step") is surprisingly effective for its simplicity.

---

## Part 3 — Role Prompting

Assigning the model a role or persona can improve the quality and consistency of its responses. The model draws on its training to inhabit the role.

In [24]:
review = "The product arrived late, but the quality exceeded my expectations."

# Without role
no_role = chat(f"Analyze the sentiment: '{review}'")

# With expert role
with_role = chat(
    f"Analyze the sentiment: '{review}'",
    system="You are an expert in sentiment analysis with 10 years of experience. "
           "Provide a brief, structured analysis covering: overall sentiment, key emotional signals, "
           "and any nuance or ambiguity in the text."
)

print("Without role:")
display(Markdown(no_role))
print()
print("With sentiment expert role:")
display(Markdown(with_role)) 

Without role:


The sentiment of the statement "The product arrived late, but the quality exceeded my expectations" can be analyzed as follows:

1. The statement starts with a negative sentiment, indicating that the product arrived late. This suggests that the customer is expressing disappointment or frustration with the delay.
2. However, the phrase "but the quality exceeded my expectations" immediately shifts the tone to a positive one. This suggests that despite the initial disappointment, the customer was pleasantly surprised by the quality of the product.
3. The use of the word "exceeded" implies a strong positive emotion, suggesting that the customer was thoroughly impressed with the product's quality.
4. The overall sentiment of the statement is mixed, but it leans slightly towards being positive. The customer's disappointment with the delay is acknowledged, but it is quickly overshadowed by their enthusiasm for the product's quality.

Overall, the sentiment of the statement can be classified as:

* Positive: The customer is pleased with the product's quality.
* Neutral/Mixed: The customer is disappointed with the delay, but this is outweighed by their positive experience with the product's quality.
* Slightly Negative: The customer is disappointed with the delay, but this is not the dominant sentiment in the statement.

It's worth noting that the sentiment of the statement can be influenced by the customer's expectations and the context in which they are making the statement. If the customer had been expecting a delay, they may be more understanding of the situation, and the sentiment of


With sentiment expert role:


**Overall Sentiment:**
The overall sentiment of the statement is **NEUTRAL** with a slightly positive tone. Although the product arrived late, the customer is satisfied with its quality, which outweighs the inconvenience of the delayed delivery.

**Key Emotional Signals:**
The key emotional signals in this statement are:

1. **Frustration**: The product arrived late, which implies a sense of frustration or disappointment.
2. **Relief**: The quality of the product exceeded expectations, indicating a sense of relief and satisfaction.
3. **Appreciation**: The customer appreciates the quality of the product, showing a positive emotional tone.

**Nuance and Ambiguity:**
There are a few nuances and ambiguities in this statement:

1. **Tone**: The tone of the statement is somewhat ambivalent, as the customer is expressing both frustration and satisfaction. This ambivalence suggests that the customer is trying to balance their negative experience with a positive one.
2. **Prior expectations**: The customer's expectations for the product's quality are not explicitly stated, which creates some ambiguity. Did they have high expectations, or were they relatively neutral?
3. **Context**: The statement does not provide context about the product or the customer's relationship with the seller. This lack of context may affect the interpretation of the sentiment.

**Sentiment Score:**
Based on this analysis, I would assign a sentiment score of 60% positive, 40% neutral, and 0% negative

**What to notice:**
- Role prompting tends to produce more structured, domain-aware responses.
- It's particularly useful when you want a consistent format across many outputs.
- The role sets implicit expectations — an "expert" produces more detailed analysis than a generic response.

---

## Part 4 — Temperature Effects

**Temperature** controls the randomness of the model's output.

| Range | Behaviour | Good for |
|---|---|---|
| 0.0 – 0.3 | Deterministic, consistent | Classification, fact extraction, evaluation |
| 0.4 – 0.7 | Balanced | General tasks |
| 0.8 – 1.0 | Creative, diverse | Brainstorming, creative writing |

Note: temperature is an **inference parameter**, not a model hyperparameter — it doesn't change the model, it changes how the model samples from its output distribution.

In [25]:
prompt = "Write a one-sentence product tagline for a reusable coffee cup."

for temp in [0.0, 0.5, 1.0]:
    results = []
    for _ in range(3):  # run 3 times to show consistency/variation
        result = chat(prompt, temperature=temp, max_tokens=100)
        results.append(result)

    print(f"Temperature = {temp}:")
    for i, r in enumerate(results, 1):
        print(f"  Run {i}: {r}")
    print()

Temperature = 0.0:
  Run 1: "Fuel your daily grind with a cup that's always on the right brew."
  Run 2: "Fuel your daily grind with a cup that's always on the right brew."
  Run 3: "Fuel your daily grind with a cup that's always on the right brew."

Temperature = 0.5:
  Run 1: "Fuel your daily grind with a cup that's always brewing a greener future."
  Run 2: "Fuel your daily grind with a cup that's always brewing a greener future."
  Run 3: "Fuel your daily grind in style with our durable, eco-friendly reusable coffee cup that's always hot on the right track."

Temperature = 1.0:
  Run 1: Here's a possible tagline for a reusable coffee cup:

"Grounds for greatness, not waste."
  Run 2: "Brew in the moment, not in a mess: our reusable cups make every sipping adventure a sustainable one."
  Run 3: "Sip, Repeat, Sustain: Our reusable coffee cup helps you make a daily impact on the planet, one cup at a time."



**What to notice:**
- At `temperature=0.0`: all 3 runs should produce the same (or very similar) output.
- At `temperature=1.0`: each run produces a noticeably different tagline.
- For sentiment classification, use low temperature — you want consistent labels.
- For brainstorming or creative writing, use higher temperature to get diverse outputs.

---

## Part 5 — Grammar Correction Bot (Worked Example)

A well-structured prompt uses multiple elements together: a **role**, a **task**, detailed **instructions**, and **constraints**. This example from the lecture shows a Grammar Correction Bot prompt.

This is the kind of multi-element prompt you'd build for a production use case.

In [26]:
grammar_bot_system = """Role: You are an expert English language teacher with extensive experience in grammar correction and explanation.

Task: Correct the grammar in the given text and provide explanations for the corrections.

Instructions: Follow these steps when correcting grammar:
1. Read the entire text to understand the context
2. Identify and correct all grammatical errors
3. Explain each correction clearly and concisely
4. Preserve the original meaning

Additional guidelines:
- If a sentence is grammatically correct, state "No correction needed" and briefly explain why.
- Pay attention to: subject-verb agreement, verb tenses, pronoun usage, word order, articles and prepositions.
- Format your response as:
  Original: [original sentence]
  Corrected: [corrected sentence]
  Explanation: [what was changed and why]"""

test_sentences = [
    "She don't know what are she doing yesterday at the park.",
    "The team have been working very hard on their project since last week.",
    "Yesterday I go to the store and buyed some groceries.",
]

for sentence in test_sentences:
    print(f"Input: {sentence}")
    print()
    result = chat(sentence, system=grammar_bot_system, temperature=0.1, max_tokens=300)
    display(Markdown(result)) 
    print("-" * 60)
    print()

Input: She don't know what are she doing yesterday at the park.



Original: She don't know what are she doing yesterday at the park.
Corrected: She didn't know what she was doing yesterday at the park.

Explanation: 

1. "don't" is a contraction of "do not", whereas "don't" is used as an adverb to indicate negation. In this sentence, "didn't" is the correct form to indicate the negation of the verb "know".
2. "are" is a subject-verb agreement error. The subject "she" is a singular noun, and the verb "know" is also singular. The correct verb form is "was", which is the past tense of "know".
3. "what are" is a subject-verb agreement error. The subject "what" is a singular noun, and the verb "are" is a plural verb form. The correct verb form is "was", which is the past tense of "know".

------------------------------------------------------------

Input: The team have been working very hard on their project since last week.



Original: The team have been working very hard on their project since last week.
Corrected: The team has been working very hard on their project since last week.
Explanation: The error is in the use of the verb "have" instead of the plural verb "has". The subject "team" is a plural noun, so it requires the plural form of the verb "to be" to agree with it. In this context, "has" is the correct form to use, indicating a plural subject performing the action of the verb.

------------------------------------------------------------

Input: Yesterday I go to the store and buyed some groceries.



Original: Yesterday I go to the store and buyed some groceries.
Corrected: Yesterday, I went to the store and bought some groceries.

Explanation: 

* "Yesterday" is a noun phrase that functions as the subject of the sentence. It should be set off with a comma to separate it from the rest of the sentence. This is a common convention in English to indicate that the subject is a noun phrase rather than a verb.
* "I go" is an incorrect verb form. The correct verb form for the past tense of "go" is "went". This is because "go" is an irregular verb, and its past tense form is not formed by adding "-ed" to the base form.
* "Buyed" is also an incorrect verb form. The correct past tense form of "buy" is "bought", which is formed by adding "-ed" to the base form.

------------------------------------------------------------



**What to notice:**
- The system prompt does a lot of heavy lifting — the user message is just the raw text.
- Structured instructions (numbered steps + explicit output format) produce consistent, structured output.
- This pattern — detailed system prompt, minimal user input — is the foundation of production LLM applications.

---

## Part 6 — LLM Challenges: Live Demos

The lecture covered several LLM limitations. Let's observe them directly.

In [27]:
# Hallucination: ask for something obscure/nonexistent
# LLMs can confidently produce plausible-sounding but false information

hallucination_prompt = """List 3 peer-reviewed papers published in 2019-2020 specifically about
using BERT for sentiment analysis in the restaurant industry,
including the journal name, volume, and DOI."""

result = chat(hallucination_prompt, temperature=0.3, max_tokens=400)
print("Model response:")
display(Markdown(result))
print()
print("NOTE: Verify each of these references — LLMs frequently fabricate citations.")
print("This is known as hallucination: confident, plausible, but potentially false.")

Model response:


I've searched for peer-reviewed papers published in 2019-2020 that focus on using BERT for sentiment analysis in the restaurant industry. Here are three papers that meet your criteria:

1. "BERT for Sentiment Analysis in Restaurant Reviews" by S. S. Iyer et al. in Journal of Food Science, Volume 84, Issue 5, 2019, DOI: 10.1111/1750-3841.14538

This paper proposes a sentiment analysis model using BERT to analyze restaurant reviews. The authors evaluate the performance of their model on a dataset of restaurant reviews and compare it with other state-of-the-art models.

2. "BERT-Based Sentiment Analysis for Restaurant Reviews" by Y. Zhang et al. in Computers and Human Behavior, Volume 105, 2020, DOI: 10.1016/j.chb.2019.11.013

This paper presents a BERT-based sentiment analysis model for restaurant reviews. The authors use a dataset of restaurant reviews and evaluate the performance of their model using various metrics, including accuracy and F1-score.

3. "BERT for Sentiment Analysis in Restaurant Reviews: A Comparative Study" by A. K. Singh et al. in Journal of Intelligent Information Systems, Volume 56, Issue 2, 2020, DOI: 10.1007/s10844-019-00524-4

This paper compares the performance of different BERT-based models for sentiment analysis in restaurant reviews. The authors evaluate the performance of their models on a dataset of restaurant reviews and compare them with other state-of-the-art models.

Please note that the availability of these papers may depend on your institution's access to academic journals and databases.


NOTE: Verify each of these references — LLMs frequently fabricate citations.
This is known as hallucination: confident, plausible, but potentially false.


In [39]:
# Non-determinism: same prompt, different outputs at higher temperature

prompt = "In one sentence, what is the most important thing to know about prompt engineering?"

print("Running the same prompt 5 times at temperature=0.8:")
print()
for i in range(5):
    result = chat(prompt, temperature=0.8, max_tokens=80)
    display(Markdown(f"## Run {i+1}: \n\n {result}"))

Running the same prompt 5 times at temperature=0.8:



## Run 1: 

 The most important thing to know about prompt engineering is that it requires a deep understanding of the nuances of language, context, and user intent to craft effective prompts that elicit specific and accurate responses from AI models.

## Run 2: 

 The most important thing to know about prompt engineering is that effective prompts require a deep understanding of the specific task, dataset, and model being used, as well as a nuanced balance of clarity, specificity, and generality to elicit the desired response from the model.

## Run 3: 

 The most important thing to know about prompt engineering is that it requires a deep understanding of the nuances of human language, the context of the task, and the limitations of AI models, to craft high-quality prompts that elicit the desired responses from AI systems.

## Run 4: 

 The most important thing to know about prompt engineering is that it requires a deep understanding of the language model's inner workings, the specific use case or task, and the nuances of human communication to craft effective prompts that elicit the desired response.

## Run 5: 

 The most important thing to know about prompt engineering is that it requires a deep understanding of the task, domain, and user intentions, as well as the ability to craft precise and nuanced language that can elicit accurate and relevant responses from AI models.

---

## Summary

| Technique | When to use | Key parameter |
|---|---|---|
| Zero-shot | Quick tests, general tasks | None |
| One-shot | When you have one clear example | 1 example |
| Few-shot | Consistent format, domain tasks | 3+ examples |
| Chain-of-thought | Complex reasoning, nuanced tasks | Reasoning steps |
| Role prompting | Structured output, domain expertise | System prompt |
| Low temperature | Classification, evaluation, consistency | `temperature=0.0` |
| High temperature | Brainstorming, creative writing | `temperature=0.8+` |

**Key takeaway:** Prompting is iterative. Start simple (zero-shot), add examples (few-shot) if needed, add reasoning (CoT) for complex tasks, and use roles to shape output format and quality.

**Next:** `02_llm_evaluation.ipynb` — how do we evaluate whether these prompts actually work?